In [ ]:
%load_ext autoreload
%autoreload 2
from manager_tenure import features
from manager_tenure.config import RAW_DATA_DIR
from manager_tenure.dataset import find_latest_snapshot
snap = find_latest_snapshot(RAW_DATA_DIR)

import pandas as pd
import janitor

In [ ]:
snap = find_latest_snapshot(RAW_DATA_DIR)
spells_table = pd.read_html(snap, match = "Duration")[0]
spells_table = spells_table.drop(columns=["Nat.", "Years in League", "Ref."])
spells_table = spells_table.clean_names(strip_underscores=True)
spells_table


# Exploring the Premier League manager spells

This notebook explores the **cleaned** table that `make data` produces (`data/processed/spells.parquet`).
Each row is one managerial spell: one manager at one club, from `start_date` to `end_date`.

The sections are:
1. Load the table and check its shape
2. Quick looks at the data and its quirks
3. How long do spells last?
4. A closer look at **Southampton**
5. Checking the Premier League tenure column against the raw Wikipedia labels

Run the cells in order from the top. Each code cell has a short explanation above it.

## 1. Load the table
We read the parquet file the pipeline wrote, then check its size and column types.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from manager_tenure.config import SPELLS_FILE

spells = pd.read_parquet(SPELLS_FILE)
print(f"{len(spells)} spells, {spells['name'].nunique()} different managers, {spells['club'].nunique()} clubs")
spells.dtypes

## 2. Quick looks and data quirks
`head()` shows the first rows. `describe()` gives summary statistics for the numbers (like R's `summary()`).
Then we count the things the cleaning step flagged: caretakers, ongoing spells and missing nationalities.

In [ ]:
spells.head(3)

In [ ]:
spells[["tenure_days", "pl_tenure_days", "duration_days"]].describe().round(0)

In [ ]:
# Each flag is True/False, so the sum counts the Trues.
print("Caretaker spells:  ", spells["is_caretaker"].sum())
print("Ongoing spells:    ", spells["is_ongoing"].sum())
print("Missing nationality:", spells["nationality"].isna().sum())
print("Spells with zero Premier League days:", (spells["pl_tenure_days"] == 0).sum())

## 3. How long do spells last?
Tenure is measured two ways:
- `tenure_days`: total time in charge
- `pl_tenure_days`: days in the club's Premier League seasons

Both are in days. Dividing by 365.25 gives years. The histogram compares them.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
# The tenure columns are in days, but we divide by 365.25 to get years,
# so the bins must be in years too: one bin per year, from 0 to 27 (the longest is 26.6).
bins = range(0, 28)
ax.hist(spells["tenure_days"] / 365.25, bins=bins, alpha=0.6, label="Total tenure")
ax.hist(spells["pl_tenure_days"] / 365.25, bins=bins, alpha=0.6, label="Premier League tenure")
ax.set_xlabel("Years")
ax.set_ylabel("Number of spells")
ax.set_xlim(0, 27)
ax.legend()
ax.set_title("Most spells are short: a few long ones stand out")
plt.show()

In [ ]:
# The longest spells, total tenure.
spells.nlargest(10, "tenure_days")[["name", "club", "start_date", "end_date", "tenure_days"]]

## 4. A closer look at Southampton
We filter the table to every Southampton spell. Caretaker spells are included, so the
list shows every change in charge, however short.

In [ ]:
sot = spells[spells["club"] == "Southampton"].sort_values("start_date")
print(f"{len(sot)} spells under {sot['name'].nunique()} managers")
sot[["name", "start_date", "end_date", "is_caretaker", "tenure_days", "pl_tenure_days"]]

In [ ]:
# One row per manager: how many spells, and how long they were in charge in total.
# groupby() then agg() is the pandas version of group_by() %>% summarise() in dplyr.
by_manager = (
    sot.groupby("name")
    .agg(
        spells=("name", "size"),
        total_days=("tenure_days", "sum"),
        pl_days=("pl_tenure_days", "sum"),
        first_start=("start_date", "min"),
    )
    .sort_values("total_days", ascending=False)
)
by_manager["total_years"] = (by_manager["total_days"] / 365.25).round(1)
by_manager.head(10)

In [ ]:
# A timeline: each bar is one spell. Caretaker spells are hatched, so they stand out.
fig, ax = plt.subplots(figsize=(10, 9))
for i, row in enumerate(sot.itertuples()):
    ax.barh(
        y=i,
        width=(row.end_date - row.start_date).days,
        left=row.start_date,
        height=0.7,
        color="tab:grey" if row.is_caretaker else "tab:red",
        hatch="//" if row.is_caretaker else None,
        edgecolor="white",
    )
ax.set_yticks(range(len(sot)))
ax.set_yticklabels(sot["name"])
ax.invert_yaxis()  # oldest at the top
ax.set_title("Southampton managers, 1991 to 2025 (grey hatched = caretaker)")
ax.xaxis_date()
plt.tight_layout()
plt.show()

## 5. How Premier League tenure is measured
Premier League time now comes from the match data, not from Wikipedia's "Years in League" labels.
The file `data/processed/premier_league_seasons.parquet` has one row per club per season, with the
date of its first and last match. A spell's Premier League time is the number of days that fall inside
those dates.

Here are Southampton's Premier League seasons, from the membership table:

In [ ]:
from manager_tenure.config import PREMIER_LEAGUE_SEASONS_FILE

memberships = pd.read_parquet(PREMIER_LEAGUE_SEASONS_FILE)
memberships[memberships["club"] == "Southampton"]

### Questions to think about
1. Compare Southampton's seasons above with what you remember of their history. Which seasons are missing,
   and do the gaps match relegations?
2. Summer breaks count when the club stays in the Premier League on both sides of them, but not after a
   promotion or relegation. Does that rule match how you would measure a manager's time in the league? What about a
   spell that ends in the summer, such as Mark Hughes's (2018)?
3. Which Southampton spell has the most Premier League days, and how does that compare with its total?

These questions connect to open question 8 in `docs/docs/open-questions.md`.